# Try WGF on MgkPro with New Measurements.

## Prepare Notebook.

### Import Libraries.

In [1]:
import os
import sys

import hydra
import matplotlib.pyplot as plt
import numpy
import omegaconf
import scanpy as sc
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import torch

import scopt
import sc_exp_design


### Import Additional Modules.

In [2]:
sys.path.insert(0, "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/shared_utils")
from experiment_utils import (
    get_forward_model,
    generate_with_condition,
    get_transformed_data,
    get_loss_fn,
    get_target_dict,
)
from inverse_utils import (
    map_df,
    loss_fn_factory
)

### Read Configurations.

In [3]:
with hydra.initialize(
    config_path="../../loss_guidance/config"
):
    config = hydra.compose(
        config_name="run_inverse_constrained",
        overrides=[
            "paths=new_measurements",
            "sampling.target_cell_type=MgkPro"
        ]
    )

/tmp/ipykernel_1951739/903777403.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatibility version level, or None.
Will assume defaults for version 1.1
  with hydra.initialize(


## Prepare Loss Function.

### Load Forward Model.

In [4]:
forward_model, (
    _,
    target_prediction_model
) = get_forward_model(config)
forward_model.target_prediction_model.resc_model["model"].eval()
forward_model.forward_model.velocity_field.eval()


NeuralVelocityField(
  (vf_modules): ModuleDict(
    (x_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=27, out_features=256, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): ELU(alpha=1.0)
        )
        (2): Sequential(
          (0): Linear(in_features=256, out_features=128, bias=True)
          (1): Identity()
        )
      )
    )
    (time_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=512, out_features=256, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): ELU(alpha=1.0)
        )
        (2): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): Identity()
        )
      )
    )
    (condition_encoder): C

### Prepare Label Encoder for Cell Types.

In [5]:
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = ct_le.classes_.tolist()


### Prepare Loss Function.

In [6]:
loss_fns = get_loss_fn(config)

target = get_target_dict(
    config,
    classes,
    forward_model.forward_model.device
)

non_linearity = torch.nn.Identity()

compute_loss, noise = loss_fn_factory(
    loss_fns,
    config,
    target,
    non_linearity,
    forward_model,
)

## Run Optimization.

### Load Prior Models.

In [7]:
# define paths to model checkpoints
prior_cfm_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/checkpoints/prior/flow_matching/2026-03-24_13-24-10/divine-grass-17_FlowMatchingWithScore.pkl"
prior_fm_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/checkpoints/prior/flow_map/2026-03-24_16-31-51/swept-morning-20_FlowMap.pkl"

# initialize prior models
prior_cfm = sc_exp_design.models.FlowMatchingWithScore.load(prior_cfm_path)
prior_fm = sc_exp_design.models.FlowMap.load(prior_fm_path)


### Prepare Optimization Settings.

### Initialize Posterior Sampler.

In [8]:
# set sampler parameters
estimator = "hutch-rademacher"
gamma = 1.0
horizon = 1.0

# prepare time rescaling and posterior sampler
time_warping = scopt.schedulers.CosineWarping(horizon=horizon)
posterior_sampler = scopt.methods.ULAGuidedFlow(
    compute_loss,
    prior_cfm,
    flow_map=prior_fm,
    scheduler=time_warping,
    gamma=gamma,
    estimator=estimator,
    use_flow_map_score_estimator=False,
)


### Sample From Posterior.

In [ ]:
# optimization params
num_opt_samples = 10
num_time_steps = 100
solver_kwargs = {
    "method": "euler",
    "atol": 1e-5,
    "rtol": 1e-5,
}

sampling_traj = posterior_sampler.sample(
    num_opt_samples,
    num_time_steps=num_time_steps,
    solver_kwargs=solver_kwargs,
).detach().cpu().numpy()
opt_samples = sampling_traj[-1]
torch.cuda.empty_cache()

OutOfMemoryError: CUDA out of memory. Tried to allocate 36.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 19.00 MiB is free. Including non-PyTorch memory, this process has 31.71 GiB memory in use. Of the allocated memory 30.79 GiB is allocated by PyTorch, and 565.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)